In [8]:
!pip install datasets transformers sentence-transformers evaluate rouge-score nltk spacy torch torchvision torchaudio scikit-learn pandas matplotlib seaborn

In [9]:
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [10]:
import pandas as pd
import numpy as np
import re
import nltk
import spacy
import torch

from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from datasets import load_dataset

In [11]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
nlp = spacy.load("en_core_web_sm")

stop_words = set(stopwords.words('english'))

In [13]:
df = pd.read_csv("/content/BBCNews.csv")

df.head()

,Unnamed: 0,descr,tags
0,0,chelsea sack mutu chelsea have sacked adrian ...,"sports, stamford bridge, football association,..."
1,1,record fails to lift lacklustre meet yelena i...,"sports, madrid, birmingham, france, scotland, ..."
2,2,edu describes tunnel fracas arsenals edu has ...,"sports, derby, brazil, tunnel fracasedu, food,..."
3,3,ogara revels in ireland victory ireland flyha...,"sports, bbc, united kingdom, ireland, brian o'..."
4,4,unclear future for striker baros liverpool fo...,"sports, liverpool, daily sport, millennium sta..."


In [14]:
df.drop(columns=["Unnamed: 0"], inplace=True)

df.head()

,descr,tags
0,chelsea sack mutu chelsea have sacked adrian ...,"sports, stamford bridge, football association,..."
1,record fails to lift lacklustre meet yelena i...,"sports, madrid, birmingham, france, scotland, ..."
2,edu describes tunnel fracas arsenals edu has ...,"sports, derby, brazil, tunnel fracasedu, food,..."
3,ogara revels in ireland victory ireland flyha...,"sports, bbc, united kingdom, ireland, brian o'..."
4,unclear future for striker baros liverpool fo...,"sports, liverpool, daily sport, millennium sta..."


In [15]:
df = df.rename(columns = {"descr" : 'text',"tags" : 'category'})

In [16]:
df.head()

,text,category
0,chelsea sack mutu chelsea have sacked adrian ...,"sports, stamford bridge, football association,..."
1,record fails to lift lacklustre meet yelena i...,"sports, madrid, birmingham, france, scotland, ..."
2,edu describes tunnel fracas arsenals edu has ...,"sports, derby, brazil, tunnel fracasedu, food,..."
3,ogara revels in ireland victory ireland flyha...,"sports, bbc, united kingdom, ireland, brian o'..."
4,unclear future for striker baros liverpool fo...,"sports, liverpool, daily sport, millennium sta..."


In [17]:
df['category'] = df['category'].apply(
    lambda x: str(x).split(',')[0].strip()
)

In [18]:
print(df['category'].unique())

['sports' 'education' 'sunderland' 'human interest' 'player' 'coach'
 'australia' 'hospitality' 'glaxosmithkline' 'politics' 'mps' 'business'
 'disaster' 'environment' 'social issues' 'bbc'
 "the 400th anniversary of guy fawkes' conspiracy"
 'alcohol focus scotland' 'law' 'united kingdom' 'london' 'health'
 'mr brown' 'war' 'labor' 'anniversary of the train bombings' 'religion'
 'las vegas' 'melbourne' 'athens' 'n hines' 'bath' 'edinburgh'
 'technology' 'icstis' 'crystal cruises' 'entertainment'
 'the 10th anniversary of the pc pro awards' 'sony' 'pew' 'the star'
 'merit-free heavy metal soundtrack' 'capgemini' 'exposition' 'iraq'
 'new york' 'jedi' 'hewlett packard' 'goldeneye' 'newcastle' 'amazon.com'
 'santos' 'belfast' 'gamer'
 'the uk are celebrating their 20th anniversary' 'rising sun' 'nintendo'
 "the net's most iconic companies" 'xinhua news agency' 'brazil' 'google'
 'washington' 'the anniversary of the liberation of auschwitz' 'scotland'
 'melton mowbray' 'manchester'
 'the 2

In [19]:
allowed_categories = [
    'sports',
    'business',
    'politics',
    'technology',
    'entertainment'

]

df = df[
    df['category'].isin(allowed_categories)
]

df = df.reset_index(drop=True)

In [20]:
print(df['category'].value_counts())

category
sports           475
entertainment    413
business         399
technology       391
politics         294
Name: count, dtype: int64


In [21]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^a-zA-Z ]", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()


def preprocess_text(text):

    text = clean_text(text)

    doc = nlp(text)

    tokens = []

    for token in doc:

        if (
            token.text not in stop_words and not token.is_punct
        ):
            tokens.append(token.lemma_)

    return " ".join(tokens)

    #Stop Word Removal: Deletes common words that don't add much meaning
     #(like "the," "is," or "at").Lemmatization: Reduces words to their base or
    #  dictionary form (e.g., "running" becomes "run," "mice" becomes "mouse").

In [22]:
df['processed_text'] = df['text'].apply(preprocess_text)

df.head()

,text,category,processed_text
0,chelsea sack mutu chelsea have sacked adrian ...,sports,chelsea sack mutu chelsea sack adrian mutu fai...
1,record fails to lift lacklustre meet yelena i...,sports,record fail lift lacklustre meet yelena isinba...
2,edu describes tunnel fracas arsenals edu has ...,sports,edu describe tunnel fracas arsenal edu lift li...
3,ogara revels in ireland victory ireland flyha...,sports,ogara revel ireland victory ireland flyhalf ro...
4,unclear future for striker baros liverpool fo...,sports,unclear future striker baros liverpool forward...


In [23]:
x = df['processed_text']
y = df['category']

x_train, x_temp, y_train, y_temp = train_test_split(
    x, y, test_size=0.3, random_state=42, stratify = y)

x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=42, stratify = y_temp)

print(len(x_train), len(x_val), len(x_test))

1380 296 296


In [24]:
vectorizer = TfidfVectorizer( max_features=10000)

x_train_tfidf = vectorizer.fit_transform(x_train)

x_val_tfidf = vectorizer.transform(x_val)

x_test_tfidf = vectorizer.transform(x_test)


In [25]:
lr_model = LogisticRegression()
lr_model.fit(x_train_tfidf, y_train)

LogisticRegression()

In [26]:
val_preds = lr_model.predict(x_val_tfidf)

print(classification_report(y_val, val_preds))

               precision    recall  f1-score   support

     business       0.95      0.98      0.97        60
entertainment       1.00      0.95      0.98        62
     politics       0.98      0.95      0.97        44
       sports       1.00      1.00      1.00        71
   technology       0.97      1.00      0.98        59

     accuracy                           0.98       296
    macro avg       0.98      0.98      0.98       296
 weighted avg       0.98      0.98      0.98       296



In [27]:
test_preds = lr_model.predict(x_test_tfidf)

print(classification_report(y_test, test_preds))

               precision    recall  f1-score   support

     business       0.95      0.90      0.92        60
entertainment       0.95      0.94      0.94        62
     politics       0.91      0.93      0.92        44
       sports       0.97      0.97      0.97        72
   technology       0.92      0.97      0.94        58

     accuracy                           0.94       296
    macro avg       0.94      0.94      0.94       296
 weighted avg       0.94      0.94      0.94       296



In [28]:
ag_news = load_dataset("SetFit/ag_news")

print(ag_news)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train.jsonl:   0%|          | 0.00/33.8M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 7600
    })
})


In [29]:
train_df = pd.DataFrame(ag_news['train'])

test_df = pd.DataFrame(ag_news['test'])

train_df.head()

train_df.drop(columns=["label_text"], inplace=True)

train_df.head()

print(train_df['label'].unique())

[2 3 1 0]


In [30]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'],
    train_df['label'],
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']
)

In [31]:
train_texts = train_texts.apply(preprocess_text)

val_texts = val_texts.apply(preprocess_text)

test_df['processed_text'] = test_df['text'].apply(preprocess_text)

In [32]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [33]:
MAX_WORDS = 20000

MAX_LEN = 200

tokenizer = Tokenizer(num_words=MAX_WORDS)

tokenizer.fit_on_texts(train_texts)

In [34]:
X_train_seq = tokenizer.texts_to_sequences(train_texts)

X_val_seq = tokenizer.texts_to_sequences(val_texts)

X_test_seq = tokenizer.texts_to_sequences(
    test_df['processed_text']
)

In [35]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN
)

STEP 21: CONVERT TO TENSORS

In [36]:
X_train_tensor = torch.tensor(X_train_pad)

X_val_tensor = torch.tensor(X_val_pad)

X_test_tensor = torch.tensor(X_test_pad)

y_train_tensor = torch.tensor(train_labels.values)

y_val_tensor = torch.tensor(val_labels.values)

y_test_tensor = torch.tensor(test_df['label'].values)

In [37]:
from torch.utils.data import TensorDataset, DataLoader

In [38]:
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

In [39]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64
)

In [40]:
import torch.nn as nn

In [41]:
class LSTMClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        output_dim
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(
            hidden_dim,
            output_dim
        )

    def forward(self, x):

        embedded = self.embedding(x)

        output, (hidden, cell) = self.lstm(embedded)

        hidden = self.dropout(hidden[-1])

        return self.fc(hidden)

In [42]:
VOCAB_SIZE = MAX_WORDS

EMBED_DIM = 128

HIDDEN_DIM = 256

OUTPUT_DIM = 4

In [43]:
model = LSTMClassifier(
    VOCAB_SIZE,
    EMBED_DIM,
    HIDDEN_DIM,
    OUTPUT_DIM
)

In [1]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.05
)

NameError: name 'nn' is not defined

In [ ]:
EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}")

    print(f"Loss: {total_loss:.4f}")

In [ ]:
model.eval()

all_preds = []
all_labels = []
